In [0]:
# =============================================================================
# Notebook: 01_setup_source
# Purpose: Simulates an external source database generating inventory shipments.
# =============================================================================
from pyspark.sql.types import *
import random
from datetime import datetime, timedelta

# Create the path for our "Mock Source Database" inside the system container
storage_account = "stsupplychaindata"
mock_source_path = f"abfss://system@{storage_account}.dfs.core.windows.net/mock_source_db/"

# Generate 1000 initial shipments from January 2024
start_date = datetime(2024, 1, 1)
data = []
for i in range(1, 1001):
    shipment_date = start_date + timedelta(days=random.randint(0, 30), hours=random.randint(0, 23))
    data.append((
        i,                                      # shipment_id
        f"WH-{random.randint(10, 50)}",         # warehouse_id
        f"SKU-{random.randint(100, 999)}",      # product_sku
        random.randint(50, 500),                # quantity_shipped
        shipment_date                           # created_at (THIS IS OUR WATERMARK COLUMN)
    ))

schema = StructType([
    StructField("shipment_id", IntegerType(), True),
    StructField("warehouse_id", StringType(), True),
    StructField("product_sku", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("created_at", TimestampType(), True)
])

df_source = spark.createDataFrame(data, schema)
df_source.write.mode("overwrite").format("delta").save(mock_source_path)

print(f"Mock Source Database created with {df_source.count()} baseline shipments.")
df_source.show(5)


Mock Source Database created with 1000 baseline shipments.
+-----------+------------+-----------+--------+-------------------+
|shipment_id|warehouse_id|product_sku|quantity|         created_at|
+-----------+------------+-----------+--------+-------------------+
|          1|       WH-45|    SKU-651|     284|2024-01-12 13:00:00|
|          2|       WH-41|    SKU-357|     265|2024-01-07 01:00:00|
|          3|       WH-26|    SKU-417|     349|2024-01-23 02:00:00|
|          4|       WH-15|    SKU-555|     416|2024-01-24 10:00:00|
|          5|       WH-44|    SKU-755|     253|2024-01-27 07:00:00|
+-----------+------------+-----------+--------+-------------------+
only showing top 5 rows
